# Credit Card SQL Chat (HTML + Excel)

This notebook indexes the credit-card HTML + Excel pair into SQLite and answers chat queries from the SQL-backed underlying data.

In [3]:
%pip install --upgrade pip setuptools wheel
%pip install openpyxl beautifulsoup4 lxml ipywidgets

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import sqlite3
import sys

try:
    import ipywidgets as widgets
    from IPython.display import display
except Exception as exc:
    raise RuntimeError('Install ipywidgets: pip install ipywidgets') from exc

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.creditcard_indexer import build_creditcard_index
from src.creditcard_query import creditcard_chat_turn


In [5]:
HTML_PATH = ROOT / '_references' / 'creditcard' / 'creditcard.html'
EXCEL_PATH = ROOT / '_references' / 'creditcard' / 'Sample Ledger Credit Card Updated.xlsx'
DB_PATH = ROOT / 'data' / 'creditcard_index.db'

build_creditcard_index(str(EXCEL_PATH), str(HTML_PATH), str(DB_PATH), report_id='creditcard', rebuild=True)

conn = sqlite3.connect(str(DB_PATH))
src = conn.execute('SELECT report_id, sheet_name, row_count, html_column_count, excel_column_count, header_match FROM report_sources').fetchall()
sample_rows = conn.execute('SELECT row_num, date, payee_name, category, tag, charge, payment FROM transactions ORDER BY row_num LIMIT 5').fetchall()
conn.close()

print('Index DB:', DB_PATH)
print('report_sources:', src)
print('sample rows:')
for r in sample_rows:
    print(r)

Index DB: /Users/atheeshkrishnan/AK/DEV/infoapp-dynamic-retrieval/data/creditcard_index.db
report_sources: [('creditcard', 'Sample Ledger Credit Card CVS3', 604, 17, 17, 1)]
sample rows:
(1, '2022-01-01', 'Cube Eatery', 'Meals & Entertainment', 'Lunch ', 86.2488, None)
(2, '2022-01-01', 'Ancient Steak House', 'Meals & Entertainment', 'Lunch ', 87.2742, None)
(3, '2022-01-02', 'Backyard Bowls', 'Meals & Entertainment', 'Lunch ', 50.175799999999995, None)
(4, '2022-01-03', 'The supper club', 'Meals & Entertainment', 'Lunch ', 74.906, None)
(5, '2022-01-04', 'Dinning Terris', 'Meals & Entertainment', 'Lunch ', 56.6271, None)


### Suggested prompts\n
- `show row for Cube Eatery on 2022-01-01`\n
- `total charge for March 2022`\n
- `count transactions for Lunch tag`\n
- `average payment in 2022`\n

In [6]:
question_box = widgets.Textarea(
    placeholder='Ask about the credit card data...',
    layout=widgets.Layout(width='820px', height='90px')
)
send_button = widgets.Button(description='Send', button_style='primary')
clear_button = widgets.Button(description='Clear', button_style='warning')
chat_html = widgets.HTML(value='<b>Chat</b><hr>')
evidence_accordion = widgets.Accordion(children=[widgets.HTML(value='No evidence yet.')])
evidence_accordion.set_title(0, 'Evidence')
history = []

def render_history():
    blocks = ['<b>Chat</b><hr>']
    for role, text in history:
        safe = (
            text.replace('&', '&amp;')
            .replace('<', '&lt;')
            .replace('>', '&gt;')
            .replace('\n', '<br>')
        )
        color = '#1f2937' if role == 'You' else '#111827'
        blocks.append(
            f"<div style='margin:8px 0;padding:8px 10px;border-radius:10px;background:{color};color:#fff;'>"
            f"<b>{role}:</b><br>{safe}</div>"
        )
    chat_html.value = ''.join(blocks)

def render_evidence(evidence):
    if not evidence:
        evidence_accordion.children = [widgets.HTML(value='No evidence.')]
        evidence_accordion.set_title(0, 'Evidence')
        return

    rows = []
    for i, e in enumerate(evidence, start=1):
        row_num = e.get('row_num')
        row_txt = f" | row={row_num}" if row_num is not None else ''
        rows.append(
            f"<div style='margin:6px 0;padding:6px;border:1px solid #ddd;border-radius:8px;'>"
            f"<b>{i}. {e['doc_id']}</b> | {e['chunk_type']}{row_txt}<br>"
            f"{e['snippet']}<br>"
            f"<span style='opacity:.7'>score={e['score']}</span>"
            f"</div>"
        )

    evidence_accordion.children = [widgets.HTML(value=''.join(rows))]
    evidence_accordion.set_title(0, 'Evidence')

def on_send(_):
    query = question_box.value.strip()
    if not query:
        return

    history.append(('You', query))
    result = creditcard_chat_turn(query=query, db_path=str(DB_PATH), report_id='creditcard', limit=8)
    mode_line = f"[mode={result['mode']}]"
    history.append(('Agent', mode_line + '\n' + result['answer']))

    render_history()
    render_evidence(result['evidence'])
    question_box.value = ''

def on_clear(_):
    history.clear()
    render_history()
    render_evidence([])

send_button.on_click(on_send)
clear_button.on_click(on_clear)
display(widgets.VBox([question_box, widgets.HBox([send_button, clear_button]), chat_html, evidence_accordion]))
render_history()
render_evidence([])